## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

STANDARD_NAME = "croissant_standard_subset"
# STANDARD_NAME = "spatial_ecological"

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
# metadata_standard=METADATA_STANDARDS['spatial_ecological']
metadata_standard = METADATA_STANDARDS[STANDARD_NAME]
orchestrator = Orchestrator(topology_name='single')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

2026-06-16 16:02:51,585 - INFO - root - PlanExecutor initialized with topology: single
2026-06-16 16:02:51,586 - INFO - root -   Players per step: 1
2026-06-16 16:02:51,586 - INFO - root -   Debate rounds: 0
2026-06-16 16:02:51,586 - INFO - root -   Player pool: ['data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-16 16:02:51,586 - INFO - root - Orchestrator initialized with topology: single
2026-06-16 16:02:51,587 - INFO - root - [ui] planning
2026-06-16 16:02:51,587 - INFO - root - ============================================================
2026-06-16 16:02:51,587 - INFO - root - GENERATING PLAN
2026-06-16 16:02:51,587 - INFO - root - Context: biota_multi
2026-06-16 16:02:51,587 - INFO - root - Context type: multi_csv
2026-06-16 16:02:51,587 - INFO - root - Classified planning type: multi_csv
2026-06-16 16:02:51,588 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06-16 16:02:51,588 - INFO - 

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan.pretty_print()

Plan Steps:
Step 0: get_field_statistics
  Rationale: Profile all resources to obtain numeric ranges and data distributions for metadata generation.
  Required Artifacts: {}
  Produced Artifacts: ['biota:stats', 'samples:stats', 'species:stats']
Step 1: get_relationships
  Rationale: Identify and validate the connections between biota, samples, and species to define recordset relationships.
  Required Artifacts: {}
  Produced Artifacts: ['context:relationships']
Step 2: analyze_spatial_temporal
  Rationale: Extract spatial bounding box and temporal coverage from the samples resource to satisfy metadata standard requirements.
  Required Artifacts: {'samples_stats': 'samples:stats'}
  Produced Artifacts: ['context:spatial_temporal']
Step 3: generate_metadata
  Rationale: Synthesize all gathered statistics, relationships, and spatial-temporal data into the final metadata record.
  Required Artifacts: {'metadata_standard': 'metadata_standard', 'biota_stats': 'biota:stats', 'samples_stats':

In [6]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME
)

2026-06-16 16:02:54,456 - INFO - root - PlanExecutor initialized with topology: single
2026-06-16 16:02:54,456 - INFO - root -   Players per step: 1
2026-06-16 16:02:54,457 - INFO - root -   Debate rounds: 0
2026-06-16 16:02:54,457 - INFO - root -   Player pool: ['data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-16 16:02:54,457 - INFO - root - Using structured output schema: CroissantStandardSubsetMetadata
2026-06-16 16:02:54,457 - INFO - root - ============================================================
2026-06-16 16:02:54,458 - INFO - root - STARTING PLAN EXECUTION
2026-06-16 16:02:54,458 - INFO - root - Context: biota_multi
2026-06-16 16:02:54,458 - INFO - root - Type: multi_csv
2026-06-16 16:02:54,458 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06-16 16:02:54,459 - INFO - root - Steps: 4
2026-06-16 16:02:54,459 - INFO - root - =========================================================

In [7]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'A dataset containing biological records linked to sampling '
                'locations in the Wadden Sea region, covering species '
                'occurrences and sample metadata collected between 2008 and '
                '2021.',
 'filesets': [{'excludes': [], 'includes': ['biota_data.csv']},
              {'excludes': [], 'includes': ['samples_data.csv']},
              {'excludes': [], 'includes': ['species_data.csv']}],
 'inLanguage': ['en'],
 'keywords': ['Wadden Sea', 'biota', 'species', 'marine biology', 'sampling'],
 'name': 'Biota and Sample Dataset of the Wadden Sea',
 'recordsets': [{'annotation': 'Central fact table containing biological '
                               'records linked to samples and species.',
                 'data': [],
                 'examples': [],
                 'field': {'arrayShape': None,
                           'dataType': 'table',
                           'isArray': False,
                           'references': No

In [8]:
result.final_workspace

{'metadata_standard': '{\n    "name": "Name of the dataset.",\n    "description": "Explain what the dataset contains.",\n    "keywords": "List search terms or tags.",\n    "filesets": "One object per file or file collection.",\n    "recordsets": "One object per table or entity.",\n    "inLanguage": "Specify ISO 639 language codes appearing in the data, e.g. [\'en\'], [\'en\', \'fr\'], [\'zh-CN\'].",\n    "spatialCoverage": "Geographic bounding box in WGS84 with numeric fields: min_lat, min_lon, max_lat, max_lon",\n    "temporalCoverage": "Time period covered, from and to date"\n}',
 'biota:stats': '{\n  "field_statistics": {\n    "status": "error",\n    "message": "Data processing failed due to invalid JSON formatting in the source analysis output. Please re-run the statistical analysis to generate a valid schema."\n  }\n}',
 'samples:stats': '{\n  "field_statistics": {\n    "status": "error",\n    "message": "Data processing failed due to invalid JSON formatting in the source analysis